In [42]:
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')

In [43]:
df = pd.read_csv('/content/100_Unique_QA_Dataset.csv')
df

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100
...,...,...
85,Who directed the movie 'Titanic'?,JamesCameron
86,Which superhero is also known as the Dark Knight?,Batman
87,What is the capital of Brazil?,Brasilia
88,Which fruit is known as the king of fruits?,Mango


In [44]:
# tokenize
def tokenize(text):
  text = text.lower()
  text = text.replace('?', '')
  text = text.replace(',', '')
  text = text.replace("'", '')
  text = text.replace('.', '')
  return text.split()

In [45]:
tokenize("Who is the author of '1984'?")

['who', 'is', 'the', 'author', 'of', '1984']

In [46]:
# vocabulary
vocab = {'<UNK>' : 0}

def build_vocab(row):
    question  = row['question']
    answer = row['answer']

    tokenize_question = tokenize(question)
    tokenize_answer = tokenize(answer)

    tokenize_question.extend(tokenize_answer)

    for word in tokenize_question:
        if word not in vocab:
            vocab[word] = len(vocab)

df.apply(build_vocab,axis=1)
print('Build Vocab Successfully')

Build Vocab Successfully


In [47]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'mockingbird': 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 '1984': 67,
 'george-orwell': 68,
 'currency': 69,
 'unit

In [48]:
len(vocab)

324

In [49]:
# text to indices
def text_to_indices(text,vocab):

  indexed_text = []

  for word in tokenize(text):
    if word in vocab:
      indexed_text.append(vocab[word])
    else:
      indexed_text.append(vocab['<UNK>'])
  return indexed_text

In [50]:
text_to_indices('what is campusx?',vocab)

[1, 2, 0]

In [51]:
import torch
from torch.utils.data import Dataset , DataLoader

In [52]:
class QADataset(Dataset):

  def __init__(self,df,vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return len(self.df)

  def __getitem__(self,idx):
    row = self.df.iloc[idx]
    question = row['question']
    answer = row['answer']

    question_indexed = text_to_indices(question,self.vocab)
    answer_indexed = text_to_indices(answer,self.vocab)

    # Use the first token of the answer as the target.
    # If answer_indexed is empty, use the UNK token index.
    if not answer_indexed:
        target_answer_token = self.vocab['<UNK>']
    else:
        target_answer_token = answer_indexed[0]

    return torch.tensor(question_indexed), torch.tensor(target_answer_token, dtype=torch.long)

In [53]:
dataset = QADataset(df,vocab)

In [54]:
dataloader = DataLoader(dataset,batch_size=1,shuffle=True)

In [55]:
import torch.nn as nn

In [56]:
class MyRNN(nn.Module):

  def __init__(self,vocab_size):
    super().__init__()

    self.embedding = nn.Embedding(vocab_size,50)
    self.rnn = nn.RNN(50,64,batch_first = True)
    self.fc = nn.Linear(64,vocab_size)

  def forward(self,x):
    embedded_question = self.embedding(x)
    # The RNN processes the sequence and returns the output for each step
    # and the final hidden state. We only need the final hidden state for
    # a single prediction for the entire sequence.
    _, hidden = self.rnn(embedded_question)
    # hidden has shape (num_layers * num_directions, batch_size, hidden_size)
    # For a single-layer, unidirectional RNN, it's (1, batch_size, 64).
    # Squeeze the first dimension to get (batch_size, 64) for the linear layer.
    output = self.fc(hidden.squeeze(0))
    return output

In [57]:
learning_rate = 0.001
epochs = 25

In [58]:
model = MyRNN(len(vocab))

In [59]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)

In [60]:
for epoch in range(epochs):

  total_loss = 0
  for question,answer in dataloader:

    optimizer.zero_grad()

    #forward pass
    output = model(question)

    # loss
    loss = criterion(output,answer)

    #backward pass
    loss.backward()

    #update parameters
    optimizer.step()

    total_loss += loss.item()

  print(f'Epoch {epoch+1}/{epochs} Loss {total_loss/len(dataloader)}')

Epoch 1/25 Loss 5.825481006834242
Epoch 2/25 Loss 5.066739177703857
Epoch 3/25 Loss 4.18037322362264
Epoch 4/25 Loss 3.506763908598158
Epoch 5/25 Loss 2.9464549409018623
Epoch 6/25 Loss 2.4111636294258965
Epoch 7/25 Loss 1.9438210315174527
Epoch 8/25 Loss 1.5083858523103926
Epoch 9/25 Loss 1.1603851248820622
Epoch 10/25 Loss 0.8872566829125087
Epoch 11/25 Loss 0.680630716184775
Epoch 12/25 Loss 0.5293363283077875
Epoch 13/25 Loss 0.42254085656669405
Epoch 14/25 Loss 0.3402567797236972
Epoch 15/25 Loss 0.2797025237646368
Epoch 16/25 Loss 0.23370848728550805
Epoch 17/25 Loss 0.19647784348991182
Epoch 18/25 Loss 0.16851838702956837
Epoch 19/25 Loss 0.1465073209669855
Epoch 20/25 Loss 0.126556885904736
Epoch 21/25 Loss 0.1107445291760895
Epoch 22/25 Loss 0.09708901043567393
Epoch 23/25 Loss 0.08634521729416317
Epoch 24/25 Loss 0.07714290705819925
Epoch 25/25 Loss 0.06938273505204254


In [63]:
def predict(model, question, threshold=0.5):

  # convert question to numbers
  numerical_question = text_to_indices(question, vocab)

  # tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output = model(question_tensor)

  # convert logits to probs
  probs = torch.nn.functional.softmax(output, dim=1)

  # find index of max prob
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("I don't know")

  print(list(vocab.keys())[index])

In [64]:
predict(model, "What is the largest planet in our solar system?")

jupiter
